# Citrus: cumulative 300,000 draft examples
Target: 90,000 rewritten educational knowledge examples and 210,000 practical task examples. Resume the latest generated SQLite checkpoint from an attached prior notebook output. Generation may span multiple free GPU sessions. All outputs require review; counts are not quality approval.


In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1" sentencepiece


In [ ]:
import pathlib, urllib.request
# Fixed repository revision makes the notebook reproducible.
REVISION = "3d5e8bbae98d7bef517d558611e79da2fec2414a"
BASE = f"https://raw.githubusercontent.com/gentle58715-ship-it/Citrus-Training-/{REVISION}"
ROOT = pathlib.Path("/kaggle/working/citrus-generator")
ROOT.mkdir(exist_ok=True)
for name in ["generate_citrus_kaggle.py", "audit_citrus_v3.py"]:
    urllib.request.urlretrieve(f"{BASE}/scripts/{name}", ROOT / name)


In [ ]:
import sqlite3, json
OUT = pathlib.Path("/kaggle/working/citrus-300k")
OUT.mkdir(exist_ok=True)
if not (OUT / "progress.sqlite").exists():
    checkpoints = list(pathlib.Path("/kaggle/input").rglob("progress.sqlite"))
    if not checkpoints:
        raise RuntimeError("No prior generation checkpoint mounted; stop rather than restart from zero")
    choices = []
    for path in checkpoints:
        src = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
        counts = dict(src.execute("SELECT kind, COUNT(*) FROM records GROUP BY kind"))
        src.close()
        if counts.get("knowledge", 0) > 90000 or counts.get("practice", 0) > 210000:
            raise RuntimeError("Checkpoint exceeds 300k quotas")
        choices.append((sum(counts.values()), str(path)))
    best = max(choices)[1]
    src = sqlite3.connect(f"file:{best}?mode=ro", uri=True)
    dst = sqlite3.connect(OUT / "progress.sqlite")
    src.backup(dst)
    dst.close(); src.close()
    print("Resumed:", best, "rows:", max(choices)[0])


In [ ]:
import subprocess
MAX_MINUTES = 480
BASE_ARGS = ["python3", "-u", str(ROOT / "generate_citrus_kaggle.py"),
             "--out", str(OUT), "--knowledge", "90000", "--practice", "210000",
             "--extend-targets"]
try:
    with open(OUT / "generation.log", "a", buffering=1) as log:
        subprocess.run(BASE_ARGS + ["--max-minutes", str(MAX_MINUTES)],
                       stdout=log, stderr=subprocess.STDOUT, check=True)
finally:
    subprocess.run(BASE_ARGS + ["--export-only"], check=True)


In [ ]:
import json, shutil
manifest = json.loads((OUT / "manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))
subprocess.run(["python3", str(ROOT / "audit_citrus_v3.py"), "--root", str(OUT)], check=True)
archive = shutil.make_archive("/kaggle/working/Citrus-300k-progress", "zip", OUT)
print("Saved:", archive)


## Output
The JSONL is cumulative; do not concatenate prior cumulative exports again. The checkpoint preserves exact normalized duplicate checks across sessions. Semantic/factual/code review is still required. All records remain drafts with runtime_verified=false. The correct completed counts are 90,000 knowledge and 210,000 practical examples. No model training occurs here.
